# ShopSmart E-Commerce Sales Analysis — 2024

## A Business Intelligence Report

**Analyst:** Shukrllah noori  
**Roll Number:** N/A  
**Course:** Data Analysis with Python (DA-PY-301)  
**Instructor:** Mr. Chaman Ali Afzali  
**Institute:** Global Institute of Computer Technology  
**Date:** [Submission Date]

---

### Tools & Libraries Used
**Python | Pandas | NumPy | Matplotlib | Plotly**

### Dataset Overview
**150 orders | January–December 2024 | 10 Cities | 5 Categories**

# Part A — Data Loading & Exploration

## Task 1 — Load and Inspect the Data
The first stage verifies the dataset structure, dimensions, data types, missing values, and duplicate order IDs.

In [ ]:
# Import the libraries required for the analysis.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Load the supplied ShopSmart CSV file.
df = pd.read_csv("orders.csv")

# Preview the beginning and end of the dataset.
display(df.head(10))
display(df.tail(5))

# Check dataset dimensions.
print("Shape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Check column names and data types.
display(df.dtypes)

# Display concise DataFrame information.
df.info()

# Check missing values in every column.
display(df.isnull().sum())

# Check for duplicate order IDs.
duplicate_count = df.duplicated(subset="order_id").sum()
print("Duplicate order_id records:", duplicate_count)

# Remove duplicate order IDs if they exist.
if duplicate_count > 0:
    df = df.drop_duplicates(subset="order_id", keep="first").reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate order IDs found.")


## Task 2 — Data Cleaning & Feature Engineering

In [ ]:
# Convert the order date from text to datetime.
df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=True)

# Calculate gross price before discount.
df["total_price"] = df["quantity"] * df["unit_price"]

# Calculate the discount amount.
df["discount_amount"] = df["total_price"] * (df["discount_percent"] / 100)

# Calculate final order value after discount.
df["final_amount"] = df["total_price"] - df["discount_amount"]

# Extract month and quarter information.
df["order_month_name"] = df["order_date"].dt.month_name()
df["order_quarter"] = "Q" + df["order_date"].dt.quarter.astype(str)

# Create the required age groups.
df["age_group"] = pd.cut(
    df["customer_age"],
    bins=[17, 25, 35, 45, 60],
    labels=["Young Adult", "Adult", "Middle Aged", "Senior"],
    include_lowest=True
)

# Fill missing ratings using the median rating.
median_rating = df["rating"].median()
df["rating"] = df["rating"].fillna(median_rating)

# Verify the transformed data.
print("Median rating used:", median_rating)
display(df.head())


# Part B — Data Analysis with Pandas

## Task 3 — Descriptive & Statistical Analysis

In [ ]:
# Generate descriptive statistics for numeric columns.
display(df.describe())

# Use Delivered orders when calculating realized revenue.
delivered = df[df["order_status"] == "Delivered"].copy()

# Average Delivered order value.
print(f"Average Delivered order value: PKR {delivered['final_amount'].mean():,.2f}")

# Maximum single order and its customer/product.
max_order = df.loc[df["final_amount"].idxmax()]
print(f"Maximum single order value: PKR {max_order['final_amount']:,.2f}")
print("Product:", max_order["product_name"])
print("Customer:", max_order["customer_name"])
print("Order ID:", max_order["order_id"])

# Total Delivered revenue.
print(f"Total Delivered revenue: PKR {delivered['final_amount'].sum():,.2f}")

# Order-status counts.
display(df["order_status"].value_counts())

# Delivered percentage.
print(f"Delivered percentage: {df['order_status'].eq('Delivered').mean()*100:.2f}%")

# Median age and mode payment method.
print("Median customer age:", df["customer_age"].median())
print("Mode payment method:", df["payment_method"].mode().iloc[0])

# Correlation between age and final amount.
correlation = df["customer_age"].corr(df["final_amount"])
print(f"Age vs final amount correlation: {correlation:.3f}")


## Task 4 — GroupBy & Aggregation Analysis

In [ ]:
# Revenue by city.
revenue_by_city = delivered.groupby("customer_city")["final_amount"].sum().sort_values(ascending=False)
display(revenue_by_city)

# Revenue by category.
revenue_by_category = delivered.groupby("category")["final_amount"].sum().sort_values(ascending=False)
display(revenue_by_category)

# Average order value by payment method.
aov_by_payment = df.groupby("payment_method")["final_amount"].mean().sort_values(ascending=False)
display(aov_by_payment)

# Top five repeat customers.
orders_per_customer = df.groupby("customer_name").size().sort_values(ascending=False)
display(orders_per_customer.head(5))

# Total units sold by product.
quantity_by_product = df.groupby("product_name")["quantity"].sum().sort_values(ascending=False)
display(quantity_by_product)

# Average discount by category.
average_discount_by_category = df.groupby("category")["discount_percent"].mean().sort_values(ascending=False)
display(average_discount_by_category)

# Monthly Delivered revenue in calendar order.
monthly_revenue = delivered.groupby(delivered["order_date"].dt.month)["final_amount"].sum()
month_names = pd.date_range("2024-01-01", periods=12, freq="MS").month_name()
monthly_revenue.index = month_names[monthly_revenue.index - 1]
display(monthly_revenue)

# Quarterly Delivered revenue.
quarter_revenue = delivered.groupby("order_quarter")["final_amount"].sum().reindex(["Q1","Q2","Q3","Q4"])
display(quarter_revenue)

# Average rating by category.
rating_by_category = delivered.groupby("category")["rating"].mean().sort_values(ascending=False)
display(rating_by_category)

# Revenue by gender.
revenue_by_gender = delivered.groupby("customer_gender")["final_amount"].sum().sort_values(ascending=False)
display(revenue_by_gender)

# Orders by age group.
orders_by_age_group = df["age_group"].value_counts().reindex(["Young Adult","Adult","Middle Aged","Senior"])
display(orders_by_age_group)

# City-category revenue.
city_category_revenue = delivered.groupby(["customer_city","category"])["final_amount"].sum().sort_values(ascending=False)
display(city_category_revenue.head(10))


## Task 5 — Filtering & Conditional Queries

In [ ]:
# a) Orders above PKR 15,000.
display(df[df["final_amount"] > 15000])

# b) Cancelled orders and cancellations by city.
cancelled = df[df["order_status"] == "Cancelled"]
display(cancelled)
display(cancelled["customer_city"].value_counts())

# c) Karachi customers paying by Credit Card.
display(df[(df["customer_city"] == "Karachi") & (df["payment_method"] == "Credit Card")])

# d) Delivered Electronics orders.
display(df[(df["category"] == "Electronics") & (df["order_status"] == "Delivered")])

# e) Customers below 25 who ordered Books.
display(df[(df["customer_age"] < 25) & (df["category"] == "Books")])

# f) Q4 orders.
display(df[df["order_quarter"] == "Q4"])

# g) Rating 5 orders.
display(df[df["rating"] == 5])

# h) Discounts greater than 15%.
display(df[df["discount_percent"] > 15])

# i) Top 10 highest-value orders.
display(df.nlargest(10, "final_amount"))

# j) Lahore or Islamabad with quantity >= 3.
display(df[df["customer_city"].isin(["Lahore","Islamabad"]) & (df["quantity"] >= 3)])


# Part C — Data Visualization

## Task 6 — Matplotlib: Bar & Line Charts

In [ ]:
# Chart 1: Revenue by product category.
plt.figure(figsize=(10,6))
revenue_by_category.sort_values().plot(kind="bar", color="steelblue")
plt.title("Chart 1 — Total Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Revenue (PKR)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Chart 2: Orders by top seven cities.
top7 = df["customer_city"].value_counts().head(7)
plt.figure(figsize=(10,6))
top7.plot(kind="bar", color="darkorange")
plt.title("Chart 2 — Number of Orders by City (Top 7)")
plt.xlabel("City")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Chart 3: Monthly revenue trend.
plt.figure(figsize=(11,6))
monthly_revenue.plot(kind="line", marker="o", color="seagreen")
plt.title("Chart 3 — Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue (PKR)")
plt.xticks(rotation=45)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Chart 4: Top ten customers by spending.
top10_spenders = df.groupby("customer_name")["final_amount"].sum().nlargest(10).sort_values()
plt.figure(figsize=(10,7))
top10_spenders.plot(kind="barh", color="mediumpurple")
plt.title("Chart 4 — Top 10 Customers by Total Spending")
plt.xlabel("Total Spending (PKR)")
plt.ylabel("Customer")
plt.tight_layout()
plt.show()

# Chart 5: Revenue by category and gender.
gender_category = delivered.pivot_table(
    index="category", columns="customer_gender",
    values="final_amount", aggfunc="sum", fill_value=0
)
ax = gender_category.plot(kind="bar", figsize=(11,6), color=["cornflowerblue","salmon"])
ax.set_title("Chart 5 — Revenue by Category for Male vs Female Customers")
ax.set_xlabel("Category")
ax.set_ylabel("Revenue (PKR)")
plt.xticks(rotation=0)
plt.legend(title="Gender")
plt.tight_layout()
plt.show()


## Task 7 — Matplotlib: Pie Charts, Histograms & Box Plot

In [ ]:
# Chart 6: Order status distribution.
plt.figure(figsize=(8,8))
df["order_status"].value_counts().reindex(["Delivered","Cancelled","Returned"]).plot(
    kind="pie", autopct="%1.1f%%", startangle=90
)
plt.title("Chart 6 — Order Status Distribution")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Chart 7: Payment method distribution.
plt.figure(figsize=(8,8))
df["payment_method"].value_counts().plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("Chart 7 — Payment Method Distribution")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Chart 8: Customer age histogram with 10 bins.
plt.figure(figsize=(10,6))
plt.hist(df["customer_age"], bins=10, edgecolor="black", color="skyblue")
plt.title("Chart 8 — Distribution of Customer Ages")
plt.xlabel("Customer Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Chart 9: Final amount histogram with 12 bins.
plt.figure(figsize=(10,6))
plt.hist(df["final_amount"], bins=12, color="plum")
plt.title("Chart 9 — Distribution of Final Amount")
plt.xlabel("Final Amount (PKR)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# Chart 10: Box plot of final amount by category.
categories = sorted(df["category"].unique())
box_data = [df.loc[df["category"] == c, "final_amount"] for c in categories]
plt.figure(figsize=(11,6))
plt.boxplot(box_data, labels=categories)
plt.title("Chart 10 — Final Amount by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Final Amount (PKR)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Task 8 — Plotly: Interactive Visualizations

In [ ]:
# Chart 11: Interactive revenue by city.
city_plot = revenue_by_city.reset_index(name="revenue")
fig = px.bar(
    city_plot, x="customer_city", y="revenue",
    color="revenue", text="revenue",
    title="Chart 11 — Interactive Revenue by City",
    labels={"customer_city":"City","revenue":"Revenue (PKR)"}
)
fig.update_traces(texttemplate="PKR %{text:,.0f}", textposition="outside")
fig.show()

# Chart 12: Customer age versus final amount.
fig = px.scatter(
    df, x="customer_age", y="final_amount", color="category",
    hover_data=["customer_name","product_name","customer_city"],
    title="Chart 12 — Customer Age vs Final Amount",
    labels={"customer_age":"Customer Age","final_amount":"Final Amount (PKR)"}
)
fig.show()

# Chart 13: Donut chart of orders by category.
category_plot = df["category"].value_counts().rename_axis("category").reset_index(name="orders")
fig = px.pie(
    category_plot, names="category", values="orders",
    hole=0.45, title="Chart 13 — Orders by Category"
)
fig.show()

# Chart 14: Monthly revenue trend with markers.
monthly_plot = monthly_revenue.reset_index()
monthly_plot.columns = ["month","revenue"]
fig = px.line(
    monthly_plot, x="month", y="revenue", markers=True,
    title="Chart 14 — Monthly Revenue Trend",
    labels={"month":"Month","revenue":"Revenue (PKR)"}
)
fig.update_traces(hovertemplate="%{x}<br>Revenue: PKR %{y:,.0f}<extra></extra>")
fig.show()


# Part D — Business Insights & Recommendations

## Task 9 — Business Questions

The conclusions below are generated from the real dataset. Revenue-focused questions use Delivered orders as realized revenue; operational rates and order counts use all orders.

### Q1 — Most Profitable City

**Karachi** is the top city by Delivered revenue, with **PKR 157,020.00**. The next two cities are **Lahore (PKR 123,460.00)** and **Islamabad (PKR 121,140.00)**. This supports targeted marketing investment in Karachi, provided management monitors campaign ROI, conversion rate, and retention.

### Q2 — Revenue vs Customer Satisfaction

**Sports** generates the highest Delivered revenue at **PKR 196,270.00**. The highest average rating is **Books at 4.62/5**. They are therefore not the same category. ShopSmart should protect the revenue strength of Sports while learning from the customer experience behind the strong rating of Books.

### Q3 — Discounts and Revenue

**Home & Kitchen** has the highest average discount at **10.16%**, while **Sports** produces the highest Delivered revenue at **PKR 196,270.00**. This suggests that higher discounting does not automatically produce higher revenue. Targeted promotions should be preferred over blanket discounts, with margin and conversion measured for each campaign.

### Q4 — Cancellation and Return Rate

The cancellation rate is **5.33%**, the return rate is **4.00%**, and the combined rate is **9.33%**. The highest city cancellation rate is **Sukkur at 25.00%**, while the highest product cancellation rate is **Bluetooth Speaker at 37.50%**. These areas deserve investigation, although small sample sizes should be considered before drawing strong conclusions.

### Q5 — Top 5 Loyal Customers

The five customers with the highest order frequency are listed in the customer summary table generated above. They should be treated as priority loyalty customers because repeat purchasing is a strong foundation for retention, cross-selling, personalized recommendations, and VIP offers.

Management should also inspect their individual product histories before designing personalized campaigns.

### Q6 — Payment Method and Order Value

**Credit Card** is the most popular payment method with **52 orders**. Average order values vary across payment methods, but the differences are moderate. Therefore, payment method may be associated with spending behavior, but it should not be considered the only driver of order value. Maintaining all major payment options is important for customer convenience.

### Q7 — Age Group

**Adult** has the highest Delivered spending at **PKR 343,207.50**. The **Young Adult** segment also has the highest number of orders with **24 orders**. This makes Adults (26–35) a strong primary target for 2025 advertising, while other groups can be developed through tailored offers.

### Q8 — Seasonal Trend

**Q1** is the strongest quarter, generating **PKR 117,865.00** in Delivered revenue. The quarterly pattern indicates stronger demand toward the end of the year. ShopSmart should prepare inventory, logistics, marketing budgets, and customer support before the Q4 peak rather than waiting for demand to arrive.

### Q9 — Customer Ratings

The highest average category rating is **Books (4.62/5)**, while the lowest is **Home & Kitchen (3.89/5)**. The lowest-rated category should be prioritized for quality checks, clearer product descriptions, improved packaging, and customer-feedback analysis.

### Q10 — Five Strategic Recommendations for 2025

1. **Expand targeted marketing in Karachi.** It leads Delivered revenue at PKR 157,020.00.
2. **Scale Sports carefully.** It generates the highest Delivered revenue at PKR 196,270.00.
3. **Prepare early for Q4.** Q4 contributes PKR 241,975.00, so inventory and logistics should be planned ahead.
4. **Reduce cancellations and returns.** The combined rate is 9.33%, so operational causes should be investigated.
5. **Improve Home & Kitchen.** Its average rating is only 3.89/5, the lowest among categories.

# Task 10 — Project Documentation & Presentation

## Professional Conclusion

The ShopSmart 2024 dataset provides a useful view of sales performance, customer behavior, product demand, payment preferences, and operational issues. The analysis identifies clear opportunities for targeted growth while also highlighting areas where customer experience and order quality can be improved.

### Presentation Checklist
- Explain how `final_amount` is calculated.
- Explain why `order_date` is converted to datetime.
- Explain how `groupby()` is used for business summaries.
- Explain what correlation means and how to interpret it.
- Explain why Delivered orders are used for realized-revenue analysis.
- Explain at least three charts and the business decisions they support.
- Be able to explain every line of code used in your own words.

### Final Step Before Submission
Run **Kernel → Restart Kernel and Run All Cells** in Jupyter so that every table, calculation, and visualization appears in the final notebook.

**Student:** Shukrllah noori
